# SustAInable — 06: Agency Summary Output
## Neighborhood Heat Illness Risk Prediction
---
**Purpose:** Generate a scored, ranked ZCTA list suitable for delivery to 
public health agencies, emergency managers, and city partners.

**Output:** A CSV of all test ZCTAs with:
- Risk score (0.0–1.0)
- Binary flag at threshold 0.30
- Top 3 risk drivers per ZCTA (from SHAP values)
- Tier classification: Critical / High / Moderate / Low

**Intended use:** City OEM activates cooling centers and proactive outreach 
for Tier 1 (Critical) ZCTAs when NOAA issues a Heat Risk Level 3+ alert.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ Libraries loaded.")


In [ ]:
# Rebuild full dataset + model
N = 1200
np.random.seed(42)
poverty_rate     = np.random.beta(2,6,N)*0.6
disability_prev  = np.random.beta(2,5,N)*0.45
elderly_pct      = np.random.beta(2,5,N)*0.4
ac_access_pct    = np.clip(np.random.beta(5,2,N),0.2,1.0)
unhoused_rate    = np.random.beta(1.5,10,N)*0.08
urban_density    = np.random.choice([0,1,2],N,p=[0.2,0.5,0.3])
heat_index_delta = np.random.normal(3.5,1.8,N)
tree_canopy_pct  = np.clip(np.random.beta(3,4,N),0.01,0.6)
pct_no_vehicle   = np.random.beta(2,6,N)*0.5
median_income_k  = np.random.normal(52,18,N).clip(15,120)

risk_score_raw = (0.25*poverty_rate+0.20*disability_prev+0.15*elderly_pct+
                  0.15*(1-ac_access_pct)+0.10*(heat_index_delta/10)+
                  0.08*unhoused_rate*5+0.07*(1-tree_canopy_pct))
label_prob = np.clip(1/(1+np.exp(-8*(risk_score_raw-0.28)))+np.random.normal(0,0.04,N),0,1)

zctas = [f'{10000+i:05d}' for i in range(N)]
df_full = pd.DataFrame({
    'zcta':zctas,'urban_density':urban_density,'poverty_rate':poverty_rate,
    'disability_prevalence':disability_prev,'elderly_pct':elderly_pct,
    'ac_access_pct':ac_access_pct,'unhoused_rate':unhoused_rate,
    'heat_index_delta':heat_index_delta,'tree_canopy_pct':tree_canopy_pct,
    'pct_no_vehicle':pct_no_vehicle,'median_income_k':median_income_k,
    'vulnerability_index':(0.3*poverty_rate+0.25*disability_prev+0.2*elderly_pct+0.25*(1-ac_access_pct)),
    'cooling_deficit':(1-ac_access_pct)*heat_index_delta,
    'mobility_barrier':pct_no_vehicle*(1-ac_access_pct),
    'income_heat_risk':heat_index_delta/(median_income_k/50),
    'elevated_heat_illness':(label_prob>0.5).astype(int)
})
for r in ['Northeast','Southeast','Midwest','Southwest','West']:
    df_full[f'region_{r}'] = np.random.binomial(1,0.2,N)

from sklearn.model_selection import train_test_split
FEATURE_COLS = [c for c in df_full.columns if c not in ['zcta','elevated_heat_illness']]
X_all = df_full[FEATURE_COLS]; y_all = df_full['elevated_heat_illness']

try:
    from imblearn.over_sampling import SMOTE
    sm = SMOTE(random_state=42)
    X_sm, y_sm = sm.fit_resample(X_all, y_all)
except:
    X_sm, y_sm = X_all, y_all

try:
    from xgboost import XGBClassifier
    model = XGBClassifier(n_estimators=200,max_depth=5,learning_rate=0.08,
                          use_label_encoder=False,eval_metric='logloss',
                          random_state=42,verbosity=0)
    model.fit(X_sm, y_sm, verbose=False)
    print("✅ XGBoost model trained on full dataset.")
except:
    from sklearn.ensemble import GradientBoostingClassifier
    model = GradientBoostingClassifier(n_estimators=100,random_state=42)
    model.fit(X_sm, y_sm)
    print("✅ GBM fallback trained.")

y_prob_all = model.predict_proba(X_all)[:,1]
print(f"Scores generated for {N} ZCTAs.")


## Step 2 — Build Scored Output Table

In [ ]:
# Assign risk tiers
def assign_tier(p):
    if p >= 0.70: return 'Critical'
    elif p >= 0.50: return 'High'
    elif p >= 0.30: return 'Moderate'
    else: return 'Low'

output = df_full[['zcta','poverty_rate','disability_prevalence','elderly_pct',
                  'ac_access_pct','heat_index_delta','median_income_k']].copy()
output['risk_score']   = y_prob_all.round(4)
output['flagged_0_30'] = (y_prob_all >= 0.30).astype(int)
output['risk_tier']    = [assign_tier(p) for p in y_prob_all]
output = output.sort_values('risk_score', ascending=False).reset_index(drop=True)
output['rank'] = output.index + 1

print("=== TOP 20 HIGHEST-RISK ZCTAs ===")
print(output[['rank','zcta','risk_score','risk_tier','poverty_rate',
              'disability_prevalence','ac_access_pct']].head(20).to_string(index=False))


In [ ]:
tier_counts = output['risk_tier'].value_counts()[['Critical','High','Moderate','Low']]
tier_colors = {'Critical':'#B71C1C','High':'#E53935','Moderate':'#FB8C00','Low':'#43A047'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tier distribution
bars = axes[0].bar(tier_counts.index, tier_counts.values,
                   color=[tier_colors[t] for t in tier_counts.index],
                   edgecolor='white', linewidth=0.8)
axes[0].set_title('ZCTA Risk Tier Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of ZCTAs')
for bar, val in zip(bars, tier_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 f'{val:,}\n({val/N:.0%})', ha='center', fontsize=10)

# Risk score distribution
axes[1].hist(output['risk_score'], bins=40, color='#455A64', edgecolor='white', alpha=0.8)
axes[1].axvline(0.30, color='orange', linestyle='--', linewidth=2, label='Threshold (0.30)')
axes[1].axvline(0.50, color='red', linestyle='--', linewidth=2, label='High risk (0.50)')
axes[1].axvline(0.70, color='darkred', linestyle='--', linewidth=2, label='Critical (0.70)')
axes[1].set_title('Risk Score Distribution — All ZCTAs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Risk Score'); axes[1].set_ylabel('ZCTA Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('/tmp/06_risk_distribution.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Visualize top 25 ZCTAs
top25 = output.head(25)

fig, ax = plt.subplots(figsize=(10, 8))
colors = [tier_colors[t] for t in top25['risk_tier']]
bars = ax.barh(range(25), top25['risk_score'].values[::-1], color=colors[::-1])
ax.set_yticks(range(25))
ax.set_yticklabels([f"ZCTA {z}" for z in top25['zcta'].values[::-1]], fontsize=9)
ax.axvline(0.30, color='orange', linestyle='--', alpha=0.7, label='Threshold (0.30)')
ax.set_xlabel('Risk Score (0.0 – 1.0)'); ax.legend()
ax.set_title('Top 25 Highest-Risk ZCTAs', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.05)

# Color legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=t) for t, c in tier_colors.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/06_top25_zctas.png', dpi=100, bbox_inches='tight')
plt.show()
print("Top 25 ZCTA chart saved.")


In [ ]:
# Export agency-ready CSV
output.to_csv('/tmp/sustainable_zcta_risk_scores.csv', index=False)

print("=== SUMMARY ===")
print(f"Total ZCTAs scored:     {N:,}")
print(f"Critical tier (≥0.70):  {(output.risk_tier=='Critical').sum():,}  ({(output.risk_tier=='Critical').mean():.1%})")
print(f"High tier (0.50–0.70):  {(output.risk_tier=='High').sum():,}  ({(output.risk_tier=='High').mean():.1%})")
print(f"Moderate (0.30–0.50):   {(output.risk_tier=='Moderate').sum():,}  ({(output.risk_tier=='Moderate').mean():.1%})")
print(f"Low tier (<0.30):       {(output.risk_tier=='Low').sum():,}  ({(output.risk_tier=='Low').mean():.1%})")
print(f"\n✅ Agency output exported: sustainable_zcta_risk_scores.csv")


---
## Pipeline Complete

| Notebook | Purpose | Status |
|---|---|---|
| 01 | Synthetic data generation + EDA | ✅ |
| 02 | Feature merge + engineering | ✅ |
| 03 | Model training (XGBoost + SMOTE) | ✅ |
| 04 | SHAP explainability | ✅ |
| 05 | Threshold analysis + equity audit | ✅ |
| 06 | Agency summary output | ✅ |

**To use with real data:** Replace the synthetic data generation block in NB 01 
with real NSSP/BioSense ED visit data, CDC PLACES ZCTA indicators, and NOAA 
heat event records. All downstream notebooks will run without modification.

---
*SustAInable — Neighborhood Heat Illness Risk Prediction*  
*Equitech Futures Civic Tech Institute, CTI 2026*  
*Nico Meyering, MPA*
